### Imports and Model Key (OpenAI, Anthropic, Google)

In [ ]:
import pandas as pd
import json, os, re, glob
from typing import Dict, Any, List
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d")
 
import openai
from openai import OpenAI
api_key =
 
openai.api_key = api_key
openai_client = OpenAI(api_key=api_key)
 
import anthropic
from google import genai
from google.genai import types
 
anthropic_api_key = 
google_api_key    =
 
anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
google_client    = genai.Client(api_key=google_api_key)


### Definitions

In [2]:

GLOSSARY = """
Passage: The legal text under analysis; passages are parts of U.S. state statutes.
Element: One of the three parts of a legal provision—conduct, attendant circumstances, or result of conduct.
Mens Rea Category: The mental state applicable to an element (purposive, knowing, reckless, negligent, strict liability).
Novel Mens Rea Term: Any mens rea wording in the passage that does not clearly match one of the five standard categories.
Golden Rule Interpretation: Classify primarily from the statutory text itself. You may depart slightly from literal wording only to avoid absurdity or inconsistency, and you should minimize extra-textual inference.
""".strip()


CLASSIFICATION_RULES = """

1. Not every provision assigns a mental state to every element. Absence of mens rea language does NOT automatically mean strict liability.
2. Mark "Strict Liability" as true only if the text explicitly indicates that liability applies regardless of intent, fault, negligence, or other culpable mental state, or uses equivalent strict-liability language.
3. Use the statutory text as the primary basis for classification.
4. Apply the Golden Rule: you may make a minimal adjustment from literal wording only to avoid abject absurdity orinconsistencies, but do not rely on broad extra-textual interpretation.
5. Lexical cues are important indicators, but classify based on how the cue applies to the element under analysis.
6. A mens rea term may apply to one element but not others. Do not automatically project it across all elements.
7. If the passage contains a mens rea phrase that does not clearly fit purposive, knowing, reckless, negligent, or strict liability, record the exact wording as a novel mens rea term.
8. If no mens rea is expressed for this element, all five mens rea categories should be false.
12. "No mental state" is not one of the five categories to mark true/false. It is represented by all five categories being false for the element.
""".strip()

ELEMENTS = {
    "conduct": "The conduct element of the legal provision: the defendant's act or omission. Can also be called the action element.",
    "circumstance": "The attendant circumstance element of the legal provision: contextual or situational facts that accompany the offense.",
    "result": "The result element of the offense: the consequence or outcome caused by the defendant's conduct.",
}


MENSREA_STANDARDS_LIST = [
    {
        "title": "Purposive",
        "definition": (
            "Language signaling deliberate intent, volition, or conscious objective."
        ),
        "cues": [
            "with purpose",
            "designed",
            "with design",
            "intentionally",
            "with intent",
            "willfully",
            "malice",
        ],
           },
     {
        "title": "Knowing",
        "definition": (
            "Language signaling awareness, conscious recognition, or belief regarding conduct, circumstances, or practical certainty of result."
        ),
        "cues": [
            "knows",
            "with knowledge",
            "is aware",
            "believes",
            "knowingly",
        ],
           },
  {
        "title": "Reckless",
        "definition": (
            "Language signaling conscious awareness of risk together with disregard of that risk."
        ),
        "cues": [
            "recklessly",
            "with recklessness",
            "wantonly",
        ],
    },
    {
        "title": "Negligent",
        "definition": (
            "Language signaling failure to meet the standard of care a reasonable person would observe, including lack of awareness where the actor should have known."
        ),
        "cues": [
            "negligence",
            "with negligence",
            "reasonably should have known",
            "should have known",
            "reason to know",
            "carelessly",
            "carelessness",
        ],
    },
    {
        "title": "Strict Liability",
        "definition": (
            "Language signaling liability regardless of intent, negligence, or any other culpable mental state."
        ),
        "cues": [
            "strictly liable",
            "regardless of fault",
        ],
    },
]

MENSREA_STANDARDS = {d["title"]: d for d in MENSREA_STANDARDS_LIST}
MENSREA_TITLES = list(MENSREA_STANDARDS.keys())

In [3]:
## Cleaning function for paths
def code_to_path(code):
    code = code.lower()
    code = re.sub(r'[^a-zA-Z0-9]+', '-', code)
    return code.strip('-')

### Model Prompt Builder

In [4]:
def _format_mensrea_block() -> str:
    blocks = []
    for m in MENSREA_STANDARDS_LIST:
        cue_text = "; ".join(m["cues"])
        blocks.append(
            f"{m['title']}:\n"
            f"Definition: {m['definition']}\n"
            f"Lexical cues: {cue_text}"
        )
    return "\n\n".join(blocks)


def _format_json_schema_lines() -> str:
    base_lines = [f'"{title}": true/false' for title in MENSREA_TITLES]
    base_lines.append('"novel_mens_rea_terms": []')
    return ",\n  ".join(base_lines)


# System prompt for LLM
def build_system_prompt_element(element_name: str, element_description: str) -> str:
    mensrea_block = _format_mensrea_block()
    mensrea_json_lines = _format_json_schema_lines()

    return f"""
You classify legal passages for mens rea.

Your task:
Determine whether the passage contains language indicating each mens rea category for the following element only.

Element under analysis:
- {element_name}

Element description:
{element_description}

Important framing:
- Analyze only this one element.
- A provision may contain conduct, circumstance, result, all of them, or only some of them.
- A mens rea term may apply to one element and not to another.
- If no mens rea is expressed for this element, mark all five categories false.
- Do NOT mark strict liability true unless the text explicitly indicates strict liability.

Mens rea categories:
{mensrea_block}

Glossary:
{GLOSSARY}

Classification rules:
{CLASSIFICATION_RULES}

Output instructions:
1. Give a brief justification grounded in the statutory text.
2. Then provide the JSON decision.
3. For "novel_mens_rea_terms", include an array of exact phrases from the passage that express a possible mental state but do not clearly match the five standard categories.
4. If there are no such phrases, return an empty array.
5. Do not include any text outside the required format.

Format your response exactly as follows:

**Justification:**
[Brief reasoning]

**Decision:**
```json
{{
  "element": "{element_name}",
  {mensrea_json_lines}
}}
```""".strip()

    

### Route model name to provider

In [5]:
def call_model(model, prompt, msg, temperature, top_p, max_tokens=2048):
    if model.startswith(("gpt", "o1", "o3", "o4")):
        response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user",   "content": msg},
            ],
            temperature=temperature,
            top_p=top_p,
        )
        return response.choices[0].message.content
 
    elif model.startswith("claude"):
        response = anthropic_client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=prompt,
            messages=[{"role": "user", "content": msg}],
            temperature=temperature,
            # top_p removed — Anthropic allows only one of temperature/top_p
        )
        return response.content[0].text
 
    elif model.startswith("gemini"):
        response = google_client.models.generate_content(
            model=model,
            contents=msg,
            config=types.GenerateContentConfig(
                system_instruction=prompt,
                temperature=temperature,
                top_p=top_p,
            ),
        )
        return response.text
 
    else:
        raise ValueError(f"Don't know which provider handles model: {model}")

### API Call

In [6]:

def make_and_save_model_calls(prompt, messages, indices, path, temperature=0, top_p=1, model=None):
    all_messages_and_replies = []


    for i, msg in zip(indices, messages):
        if os.path.exists(f"{path}/message_{i}.txt"):
            continue

        reply = call_model(model, prompt, msg, temperature, top_p)
        print(reply)
 
        all_messages_and_replies.append({"role": "user", "content": msg})
        all_messages_and_replies.append({"role": "assistant", "content": reply})
 
        with open(f"{path}/message_{i}.txt", "w") as message_file:
            message_file.write(f"User: {msg}\n")
            message_file.write(f"Assistant: {reply}\n")
 
    with open(f"{path}/all_messages.json", "w") as json_file:
        json.dump(all_messages_and_replies, json_file, indent=4)

    csv_path = f"{path}/ids.csv"
    ids = pd.DataFrame(list(indices), columns=['id'])
    ids.to_csv(csv_path)


### Reading in CSVs / Outputing

In [10]:
# read in csv of passages
passages_path = '/Users/chris/Desktop/4 Test Offense/CSVs_legal text/DUI.csv'
passages_df = pd.read_csv(passages_path)

# derive the offense label from the input filename: "DUI.csv" -> "DUI"
offense_name = os.path.splitext(os.path.basename(passages_path))[0]
print(f"Offense: {offense_name}")

print(passages_df.columns)

# specify cols in csv ### HOW MANY ROWS ###

messages = passages_df['full legal text'][0:2]
indices = passages_df['law number'][0:2]

# parameters ### change model HERE
models = ['gpt-5.4-nano-2026-03-17', 'claude-haiku-4-5-20251001', 'gemini-3.1-flash-lite']
temperature = 1
top_p = 1

# Individual calls per code
def make_and_save_model_calls_per_code(messages, path, indices, temperature=temperature, top_p=top_p, model=None):
    for element_name, element_description in ELEMENTS.items():
        code_title = code_to_path(element_name)
        code_path = f'{path}/{code_title}'

        if not os.path.exists(code_path):
            os.makedirs(code_path)

        prompt = build_system_prompt_element(element_name, element_description)
        print(prompt)

        make_and_save_model_calls(
            prompt,
            messages,
            indices,
            code_path,
            temperature=temperature,
            top_p=top_p,
            model=model
        )
    
    params = {}
    params['temperature'] = temperature
    params['top_p'] = top_p
    params['model'] = model
    params['elements'] = list(ELEMENTS.keys())
    params['mensrea_titles'] = MENSREA_TITLES
    json.dump(params, open(f'{path}/params.json', 'w'))


# =========================
# BUILD FINAL STRUCTURED CSV
# =========================

def extract_justification_and_decision(message_txt_path):
    """
    Reads one saved message_<id>.txt file and extracts:
    - justification text
    - decision JSON
    Returns: (justification, decision_dict)
    """
    with open(message_txt_path, "r", encoding="utf-8") as f:
        text = f.read()

    justification = ""
    decision_dict = {}

    # Extract justification
    just_match = re.search(
        r"\*\*Justification:\*\*\s*(.*?)\s*\*\*Decision:\*\*",
        text,
        flags=re.DOTALL
    )
    if just_match:
        justification = just_match.group(1).strip()

    # Extract JSON block inside ```json ... ```
    json_match = re.search(
        r"```json\s*(\{.*?\})\s*```",
        text,
        flags=re.DOTALL
    )
    if json_match:
        json_str = json_match.group(1).strip()
        try:
            decision_dict = json.loads(json_str)
        except json.JSONDecodeError:
            print(f"Could not parse JSON in {message_txt_path}")

    return justification, decision_dict


def get_true_mensrea_terms(decision_dict):
    """
    Given a parsed decision JSON, returns mens rea titles where value == True,
    preserving the order in MENSREA_TITLES.
    """
    true_terms = []
    for title in MENSREA_TITLES:
        if decision_dict.get(title) is True:
            true_terms.append(title)
    return true_terms


def pad_to_three(items):
    """
    Ensures exactly 3 output slots.
    Example: ['Knowing'] -> ['Knowing', '', '']
    """
    items = items[:3]
    while len(items) < 3:
        items.append("")
    return items

def build_structured_output_csv(passages_df, base_path, indices, model=None, output_filename=None):
    """
    Creates the final structured CSV in base_path.
    Matches exactly the same laws that were processed in the main script.
    """
    # Build a default filename that includes the model
    if output_filename is None:
        if model is not None:
            safe_model = str(model).replace("/", "-")   # avoid stray path separators
            output_filename = f"structured_mensrea_output_model={safe_model}_{offense_name}.csv"
        else:
            output_filename = "structured_mensrea_output.csv"

    # Match exactly the same rows the script processed
    source_df = passages_df[passages_df["law number"].isin(indices)].copy()
    source_df = source_df.set_index("law number").loc[list(indices)].reset_index()

    source_df = source_df[["law number", "full legal text"]].copy()
    source_df.rename(columns={"full legal text": "legal extract"}, inplace=True)

    output_rows = []

    for _, row in source_df.iterrows():
        law_number = row["law number"]
        legal_extract = row["legal extract"]

        row_out = {
            "law number": law_number,
            "legal extract": legal_extract,
            "novel mens rea terms": "",
            "Conduct #1": "",
            "Conduct #2": "",
            "Conduct #3": "",
            "Result #1": "",
            "Result #2": "",
            "Result #3": "",
            "Circumstance #1": "",
            "Circumstance #2": "",
            "Circumstance #3": "",
            "Conduct Justification": "",
            "Result Justification": "",
            "Circumstance Justification": ""
        }

        for element_name in ["conduct", "result", "circumstance"]:
            message_path = os.path.join(base_path, element_name, f"message_{law_number}.txt")

            if not os.path.exists(message_path):
                print(f"Missing file: {message_path}")
                continue

            justification, decision_dict = extract_justification_and_decision(message_path)
            true_terms = pad_to_three(get_true_mensrea_terms(decision_dict))

            if element_name == "conduct":
                row_out["Conduct #1"] = true_terms[0]
                row_out["Conduct #2"] = true_terms[1]
                row_out["Conduct #3"] = true_terms[2]
                row_out["Conduct Justification"] = justification

            elif element_name == "result":
                row_out["Result #1"] = true_terms[0]
                row_out["Result #2"] = true_terms[1]
                row_out["Result #3"] = true_terms[2]
                row_out["Result Justification"] = justification

            elif element_name == "circumstance":
                row_out["Circumstance #1"] = true_terms[0]
                row_out["Circumstance #2"] = true_terms[1]
                row_out["Circumstance #3"] = true_terms[2]
                row_out["Circumstance Justification"] = justification

        output_rows.append(row_out)

    final_df = pd.DataFrame(output_rows, columns=[
        "law number",
        "legal extract",
        "novel mens rea terms",
        "Conduct #1",
        "Conduct #2",
        "Conduct #3",
        "Result #1",
        "Result #2",
        "Result #3",
        "Circumstance #1",
        "Circumstance #2",
        "Circumstance #3",
        "Conduct Justification",
        "Result Justification",
        "Circumstance Justification"
    ])

    output_path = os.path.join(base_path, output_filename)
    final_df.to_csv(output_path, index=False)
    print(f"Structured CSV saved to: {output_path}")

    return final_df

for model in models:
    base_path = f'/Users/chris/Desktop/CLASS_TEST/{timestamp}_t={temperature}_top_p={top_p}_model={model}/{offense_name}'
 
    if not os.path.exists(base_path):
        os.makedirs(base_path)
 
    make_and_save_model_calls_per_code(
        messages,
        base_path,
        indices,
        temperature=temperature,
        top_p=top_p,
        model=model
    )
 
    build_structured_output_csv(passages_df, base_path, indices, model=model)


Offense: DUI
Index(['law number', 'full legal text'], dtype='object')
You classify legal passages for mens rea.

Your task:
Determine whether the passage contains language indicating each mens rea category for the following element only.

Element under analysis:
- conduct

Element description:
The conduct element of the legal provision: the defendant's act or omission. Can also be called the action element.

Important framing:
- Analyze only this one element.
- A provision may contain conduct, circumstance, result, all of them, or only some of them.
- A mens rea term may apply to one element and not to another.
- If no mens rea is expressed for this element, mark all five categories false.
- Do NOT mark strict liability true unless the text explicitly indicates strict liability.

Mens rea categories:
Purposive:
Definition: Language signaling deliberate intent, volition, or conscious objective.
Lexical cues: with purpose; designed; with design; intentionally; with intent; willfully; mal